In [1]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.5 MB/s eta 0:00:00


In [11]:
import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig, pipeline
import torch
import gradio as gr

In [6]:
LLAMA = "meta-llama/Llama-3.2-3B-Instruct"

In [7]:
# Sign in to HuggingFace Hub

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [8]:
def build_messages(theme, columns, include_suggested):
    system_prompt = """
You are a synthetic data generator.

Generate realistic synthetic tabular data based on the user's request.

Rules:
- Return only CSV.
- Use the requested theme and columns.
- If include_suggested_columns is true, add useful extra columns relevant to the theme.
- If include_suggested_columns is false, only use the requested columns.
- Do not include explanations.
"""

    columns_text = ", ".join(columns)

    user_prompt = f"""
Theme: {theme}
Requested columns: {columns_text}
Include suggested columns: {include_suggested}
Generate 10 rows.
"""

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

In [9]:
generate_data = pipeline("text-generation", model=LLAMA, device="cuda")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [10]:
data = generate_data(build_messages("car_sales", ["brand", "model"], True))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [30]:
import pandas as pd
from io import StringIO

def generate_synthetic_table(theme, columns_text, include_suggested):
    columns = [c.strip() for c in columns_text.split(",") if c.strip()]
    messages = build_messages(theme, columns, include_suggested)

    data = generate_data(messages, max_new_tokens=300)
    csv_text = data[0]["generated_text"][-1]["content"].strip()

    df = pd.read_csv(StringIO(csv_text))
    return df


In [31]:
gr.Interface(
    fn=generate_synthetic_table,
    inputs=[
        gr.Textbox(label="Theme"),
        gr.Textbox(label="Columns (comma-separated)"),
        gr.Checkbox(label="Include suggested columns")
    ],
    outputs=gr.Dataframe(label="Synthetic Data"),
    flagging_mode="never"
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bcb3a8397762048582.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
